# Re-ranking semántico

La **búsqueda por palabras clave**, comúnmente utilizada en motores de búsqueda, prioriza la similitud de palabras entre la consulta del usuario y los documentos disponibles, pero suele ignorar la **semántica** de las palabras. Por ejemplo, no distinguiría entre "banco" como institución financiera y "banco" como asiento, ya que no comprende las diferencias contextuales y semánticas entre palabras. Por el contrario, la búsqueda semántica sí entiende estas diferencias, brindando resultados más precisos y relevantes.

Migrar completamente a sistemas de búsqueda semántica puede ser un reto para muchas empresas debido a la profunda integración de los sistemas basados en palabras clave. Una solución es **re-rankear** los resultados de la búsqueda por palabras clave usando un modelo de búsqueda semántica basado en **word embeddings**.

**Cohere Rerank** ofrece una solución de re-ranking semántico fácil de integrar en sistemas existentes, necesitando solo unas pocas líneas de código, permitiendo así una transición suave hacia métodos de búsqueda más avanzados y precisos.

Para obtener más información sobre `Cohere Rerank` y cómo puede beneficiar a tu sistema de búsqueda, te invito a visitar su [blog](https://txt.cohere.com/rerank/).


> **NOTA:** Al integrar la precisión contextual de la búsqueda semántica con la eficacia de la búsqueda por palabras clave, podemos superar las limitaciones inherentes de cada método, logrando así resultados de búsqueda más relevantes, precisos y ricos en contexto.

## Librerías

In [1]:
%%capture
!pip install mistralai

In [2]:
from mistralai import Mistral
from dotenv import load_dotenv
from langchain_classic.retrievers import BM25Retriever
from langchain_core.documents import Document

from src.langchain_docs_loader import load_langchain_docs_splitted

load_dotenv()

True

## Carga de datos

In [3]:
docs = load_langchain_docs_splitted()
len(docs)

556

## Creación de herramienta de búsqueda por palabras clave

BM25 es una función de ranking avanzada usada para clasificar documentos en sistemas de recuperación de información, basándose en su relevancia respecto a una consulta de búsqueda. A diferencia de la búsqueda por palabras clave básica, que solo considera la presencia o ausencia de palabras, BM25 calcula un score de relevancia para cada documento, teniendo en cuenta la frecuencia de aparición del término y su rareza en la colección de documentos.

In [4]:
keywordk_retriever = BM25Retriever.from_documents(docs)


def keyword_document_search(query: str, k: int) -> list[Document]:
    keywordk_retriever.k = k
    return keywordk_retriever.invoke(query)

## Búsqueda de documentos relevantes por palabras clave

In [5]:
relevant_keyword_documents = keyword_document_search(
    query="How to integrate LCEL into my Retrieval augmented generation system with a keyword search retriever?",
    k=100,
)

print("Keyword search results:")
for i, document in enumerate(relevant_keyword_documents):
    print(f"{i+1}. {document.metadata['source']}")

Keyword search results:
1. https://docs.langchain.com/oss/python/langchain/rag
2. https://docs.langchain.com/oss/python/langchain/retrieval
3. https://docs.langchain.com/oss/python/langchain/rag
4. https://docs.langchain.com/oss/python/integrations/providers/aws
5. https://docs.langchain.com/oss/python/langchain/short-term-memory
6. https://docs.langchain.com/oss/python/langchain/knowledge-base
7. https://docs.langchain.com/oss/python/langchain/component-architecture
8. https://docs.langchain.com/oss/python/langgraph/add-memory
9. https://docs.langchain.com/oss/python/learn
10. https://docs.langchain.com/oss/python/integrations/providers/all_providers
11. https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant
12. https://docs.langchain.com/oss/python/deepagents/quickstart
13. https://docs.langchain.com/oss/python/langchain/philosophy
14. https://docs.langchain.com/oss/python/integrations/providers/microsoft
15. https://docs.langchain.com/oss/python/contributin

## Re-ranking semántico de los documentos relevantes

Una vez que hemos obtenido los documentos más relevantes para nuestra consulta de búsqueda, podemos re-rankearlos usando un modelo de búsqueda semántica basado en **word embeddings**.

En este caso, utilizaremos `Cohere Rerank` para obtener los documentos más relevantes para nuestra consulta de búsqueda, re-rankeando los documentos obtenidos por BM25.

Para que `Cohere Rerank` funcione, necesitarás una cuenta de `Cohere` y un `API key`. Puedes obtener tu `API key` [aquí](https://dashboard.cohere.com/api-keys).

In [6]:
import os
mistral = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

In [ ]:
import time
from mistralai import Mistral, SystemMessage, UserMessage

query = "How to integrate LCEL into my Retrieval augmented generation system with a keyword search retriever?"

top_n = 3

messages = [
    SystemMessage(content=f"You are an expert relevance ranker. Given a list of {top_n} documents and a query, your job is to determine how relevant each document is for answering the query. Your output is JSON, which is a list of documents.  Each document has two fields, content and score.  relevance_score is from 0.0 to 100.0. Higher relevance means higher score."),
    UserMessage(content=f"Query: {query} Docs: {relevant_keyword_documents}"),
]

start = time.time()
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
response = client.chat.complete(
    model="mistral-medium",
    messages=messages,
)

print(f"Took {time.time() - start} seconds to re-rank documents with mistral-medium.")

Took 136.00766396522522 seconds to re-rank documents with mistral-medium.


In [11]:
response.choices[0].message.content

'```json\n[\n    {\n        "content": "## 2. Retrieval and generation\\n\\nRAG applications commonly work as follows:\\n\\n1. **Retrieve**: Given a user input, relevant splits are retrieved from storage using a [Retriever](/oss/python/langchain/retrieval#retrievers).\\n2. **Generate**: A [model](/oss/python/langchain/models) produces an answer using a prompt that includes both the question with the retrieved data\\n\\nNow let’s write the actual application logic. We want to create a simple application that takes a user question, searches for documents relevant to that question, passes the retrieved documents and initial question to a model, and returns an answer.\\n\\nWe will demonstrate:\\n\\n1. A RAG [agent](#rag-agents) that executes searches with a simple tool. This is a good general-purpose implementation.\\n2. A two-step RAG [chain](#rag-chains) that uses just a single LLM call per query. This is a fast and effective method for simple queries.\\n\\n### RAG agents\\n\\nOne formul

In [20]:
import json
from langchain_core.utils.json import parse_json_markdown

json_str = parse_json_markdown(response.choices[0].message.content)

# Sort the scores by highest to lowest and print
scores = json.loads(json.dumps(json_str))
sorted_data = sorted(scores, key=lambda x: x['score'], reverse=True)
print(json.dumps(sorted_data, indent=2))

[
  {
    "content": "## 2. Retrieval and generation\n\nRAG applications commonly work as follows:\n\n1. **Retrieve**: Given a user input, relevant splits are retrieved from storage using a [Retriever](/oss/python/langchain/retrieval#retrievers).\n2. **Generate**: A [model](/oss/python/langchain/models) produces an answer using a prompt that includes both the question with the retrieved data\n\nNow let\u2019s write the actual application logic. We want to create a simple application that takes a user question, searches for documents relevant to that question, passes the retrieved documents and initial question to a model, and returns an answer.\n\nWe will demonstrate:\n\n1. A RAG [agent](#rag-agents) that executes searches with a simple tool. This is a good general-purpose implementation.\n2. A two-step RAG [chain](#rag-chains) that uses just a single LLM call per query. This is a fast and effective method for simple queries.\n\n### RAG agents\n\nOne formulation of a RAG application is